In [2]:
import os
import kagglehub
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PayloadSchemaType,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

# Conexión al contenedor Docker local
client = QdrantClient(url="http://localhost:6333")
print("Colecciones activas en Qdrant:", client.get_collections())

c:\Users\User\Documents\DAVID2\practica_qdrant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Colecciones activas en Qdrant: collections=[CollectionDescription(name='catalogo_libros')]


In [3]:
print("Descargando dataset...")
path = kagglehub.dataset_download("dylanjcastillo/7k-books-with-metadata")
csv_path = os.path.join(path, "books.csv")

df = pd.read_csv(csv_path)

# Descartar filas sin sinopsis
df = df.dropna(subset=["description"]).reset_index(drop=True)
print(f"Total de libros listos para indexar: {len(df)}")
df[["title", "categories", "average_rating", "description"]].head(2)

Descargando dataset...
Total de libros listos para indexar: 6548


,title,categories,average_rating,description
0,Gilead,Fiction,3.85,A NOVEL THAT READERS and critics have been eag...
1,Spider's Web,Detective and mystery stories,3.83,A new 'Christie for Christmas' -- a full-lengt...


In [4]:
# Carga del modelo especificado en el reto
model = SentenceTransformer("intfloat/multilingual-e5-small")
vector_size = model.get_sentence_embedding_dimension()  # 384 dimensiones

COLLECTION_NAME = "catalogo_libros"

# Reiniciar la colección si existía previamente
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

# Creación de índices en el payload para filtrado eficiente
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="categories",
    field_schema=PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="average_rating",
    field_schema=PayloadSchemaType.FLOAT,
)

print(f"Colección '{COLLECTION_NAME}' lista con dimensión {vector_size}.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3042.13it/s]
C:\Users\User\AppData\Local\Temp\ipykernel_87512\2068254651.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vector_size = model.get_sentence_embedding_dimension()  # 384 dimensiones


Colección 'catalogo_libros' lista con dimensión 384.


In [5]:
BATCH_SIZE = 64
total_libros = len(df)
print("Iniciando inserción por lotes...")

for start_idx in range(0, total_libros, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_libros)
    batch_df = df.iloc[start_idx:end_idx]

    descriptions = batch_df["description"].tolist()
    embeddings = model.encode(descriptions, show_progress_bar=False)

    points = []
    for point_id, (_, row), vector in zip(batch_df.index, batch_df.iterrows(), embeddings):
        payload = {
            "title": str(row.get("title", "")),
            "description": str(row.get("description", "")),
            "categories": str(row.get("categories", "")),
            "average_rating": float(row.get("average_rating", 0.0)) if pd.notnull(row.get("average_rating")) else 0.0,
            "authors": str(row.get("authors", "")),
        }
        points.append(
            PointStruct(
                id=int(point_id),
                vector=vector.tolist(),
                payload=payload,
            )
        )

    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Progreso: {end_idx}/{total_libros}", end="\r")

print("\n¡Ingesta completada!")

Iniciando inserción por lotes...
Progreso: 6548/6548
¡Ingesta completada!


In [6]:
def buscar_libros(query: str, genero: str = None, limit: int = 5) -> pd.DataFrame:
    # 1. Vectorizar la consulta
    query_vector = model.encode(query).tolist()
    
    # 2. Configurar filtro si se proporciona un género
    query_filter = None
    if genero:
        query_filter = Filter(
            must=[FieldCondition(key="categories", match=MatchValue(value=genero))]
        )

    # 3. Búsqueda utilizando la API actual: query_points
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=query_filter,
        limit=limit,
    )

    # 4. Formatear la salida (los resultados vienen en response.points)
    resultados = []
    for hit in response.points:
        resultados.append({
            "id": hit.id,
            "score": round(hit.score, 4),
            "title": hit.payload.get("title"),
            "categories": hit.payload.get("categories"),
            "rating": hit.payload.get("average_rating"),
            "description": hit.payload.get("description")[:100] + "...",
        })
    return pd.DataFrame(resultados)

# Prueba de búsqueda
resultados_df = buscar_libros("space exploration and science fiction", limit=5)
resultados_df

,id,score,title,categories,rating,description
0,3334,0.9236,Triumph of the Darksword,Fiction,3.57,Science fiction....
1,3539,0.8979,2061,Computers,3.55,Science fiction-roman....
2,4466,0.8638,You Can Draw Star Wars,Juvenile Nonfiction,4.04,"Directions on drawing characters, starships, a..."
3,4786,0.8628,"Right Stuff, Wrong Sex",History,3.80,space program and the rise of the women's move...
4,5423,0.8505,A Short History of Nearly Everything,Science,4.20,In this book Bill Bryson explores the most int...


In [7]:
def recomendar_libros(me_gustan_ids: list, no_me_gustan_ids: list = None, limit: int = 5) -> pd.DataFrame:
    if no_me_gustan_ids is None:
        no_me_gustan_ids = []
        
    try:
        # Intento con el método recommend clásico
        hits = client.recommend(
            collection_name=COLLECTION_NAME,
            positive=me_gustan_ids,
            negative=no_me_gustan_ids,
            limit=limit,
        )
    except AttributeError:
        # Alternativa moderna con query_points y modelos de recomendación
        from qdrant_client.models import RecommendInput
        response = client.query_points(
            collection_name=COLLECTION_NAME,
            query=RecommendInput(positive=me_gustan_ids, negative=no_me_gustan_ids),
            limit=limit,
        )
        hits = response.points

    resultados = []
    for hit in hits:
        resultados.append({
            "id": hit.id,
            "score": round(hit.score, 4),
            "title": hit.payload.get("title"),
            "categories": hit.payload.get("categories"),
            "rating": hit.payload.get("average_rating"),
            "description": hit.payload.get("description")[:100] + "...",
        })
    return pd.DataFrame(resultados)

In [8]:
resultados_dostoyevsky = buscar_libros(query="Dostoevsky psychological classic Russian literature", limit=5)
display(resultados_dostoyevsky[["id", "title", "categories", "rating", "score"]])

,id,title,categories,rating,score
0,3221,Tolstoy: Anna Karenina,Literary Criticism,4.19,0.8806
1,1183,The Karamazov Brothers,Fiction,4.32,0.8770
2,827,Anna Karenina,Fiction,4.04,0.8690
3,789,Notes from Underground,Fiction,4.20,0.8690
4,4078,EUGENE ONEGIN - A NOVEL IN VERSE V.1,Literary Criticism,4.22,0.8623
